# Deep Learning for Image Analysis
### YOLO Object Detection on Pascal VOC 2012

**Authors:** Bo Fu, Yehoshua Perez Condori  
**Programme:** MSc Artificial Intelligence  
**Institution:** City St George's, University of London  
**Module:** INM705 Deep Learning for Image Analysis  
**Module Leader:** Dr Riad Ibadulla  

---

### Project Overview
Implementation of YOLOv1-style object detection using VGG16 backbone trained on Pascal VOC 2012 dataset with 20 object categories.

---

### Links
- **GitHub:** https://github.com/BoFu001/YOLO-object-detection
- **Colab Notebook:** https://drive.google.com/file/d/182m9Fadqzu_SJAwi9HJrPFqUUiMgEdhU/view?usp=sharing
- **Kaggle Notebook:** https://www.kaggle.com/code/bofu001/yolo-object-detection
- **Kaggle Dataset:** https://www.kaggle.com/datasets/huanghanchina/pascal-voc-2012
- **Wandb:** https://wandb.ai/bofu001-/YOLO-VOC2012

In [1]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [2]:
!pip install -q torchmetrics kaggle wandb

import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Education/Deep Learning for Image Analysis/YOLO-object-detection')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root added to sys.path: {PROJECT_ROOT}')

Project root added to sys.path: /content/drive/MyDrive/Colab Notebooks/Education/Deep Learning for Image Analysis/YOLO-object-detection


In [3]:
import os
print('Current working directory:', os.getcwd())

Current working directory: /content


In [4]:
import os
print('Drive mounted:', os.path.exists('/content/drive/MyDrive/'))
if os.path.exists('/content/drive/MyDrive/'):
    print('Contents of MyDrive:')
    print(os.listdir('/content/drive/MyDrive/')[:10])

# Let's also check if Colab Notebooks exists
if os.path.exists('/content/drive/MyDrive/Colab Notebooks/'):
    print('\nContents of Colab Notebooks:')
    print(os.listdir('/content/drive/MyDrive/Colab Notebooks/')[:10])

Drive mounted: True
Contents of MyDrive:
['Colab Notebooks', 'Crime-Type-Prediction-Using-Machine-Learning.gslides', 'Crime-Type-Prediction-Using-Machine-Learning.gvid', 'Untitled video.gvid', 'PXL_20260410_143859481.jpg', 'PXL_20260410_143857839.jpg', 'PXL_20260410_143854288.jpg', 'PXL_20260410_143852981.MP.jpg', 'PXL_20260410_084015272.jpg', 'PXL_20260410_084009367.MP.jpg']

Contents of Colab Notebooks:
['Education', '.ipynb_checkpoints']


In [5]:

import os
import wandb


try:
    from google.colab import userdata
    wandb_api_key = userdata.get('WANDB_API_KEY')
except Exception:
    wandb_api_key = None

if wandb_api_key:
    os.environ.pop("WANDB_MODE", None)
    wandb.login(key=wandb_api_key, relogin=False)
    print("WandB login enabled from Colab Secrets.")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY not found in Colab Secrets. WandB disabled, so no login prompt will appear.")
    print("Add WANDB_API_KEY to Colab Secrets if you want online WandB logging or sweeps.")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: y-benjamin_pc (y-benjamin_pc-city-st-george-s-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


WandB login enabled from Colab Secrets.


In [6]:
# Kaggle dataset setup for Colab local runtime
import os
from pathlib import Path

VOC_ROOT = Path('/content/VOC2012/VOC2012')
ZIP_PATH = Path('/content/pascal-voc-2012.zip')
KAGGLE_JSON = Path.home() / '.kaggle' / 'kaggle.json'

if VOC_ROOT.exists():
    print(f"Dataset already present at {VOC_ROOT}")
else:
    if not KAGGLE_JSON.exists():
        from google.colab import files
        uploaded = files.upload()
        uploaded_names = list(uploaded.keys())
        if not uploaded_names:
            raise RuntimeError("No kaggle.json uploaded.")
        first_file = uploaded_names[0]
        os.makedirs(Path.home() / '.kaggle', exist_ok=True)
        os.replace(first_file, KAGGLE_JSON)
        os.chmod(KAGGLE_JSON, 0o600)
        print(f"Kaggle credentials configured from: {first_file}")
    else:
        print("Using existing ~/.kaggle/kaggle.json")

    if not ZIP_PATH.exists():
        !kaggle datasets download -d huanghanchina/pascal-voc-2012 -p /content/
    else:
        print(f"Zip already present at {ZIP_PATH}")

    !unzip -qo /content/pascal-voc-2012.zip -d /content/VOC2012
    print("Dataset prepared at /content/VOC2012/VOC2012")

Dataset already present at /content/VOC2012/VOC2012


In [7]:
import sys

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Education/Deep Learning for Image Analysis/Training/YOLO-object-detection/YOLO-object-detection"
modules_dir = f"{PROJECT_ROOT}/modules"

# Ensure the updated PROJECT_ROOT is in sys.path
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [8]:
print(os.path.exists(f"{PROJECT_ROOT}/my_config.py"))
print(os.path.exists(f"{PROJECT_ROOT}/modules"))

True
True


In [9]:
import random
import numpy as np
import torch
import io
import os
import re
import json
import wandb
import sys
import pandas as pd
from IPython.display import display


In [10]:
from my_config import SEED, CKPT_DIR, DEVICE, IMG_DIR, ANN_DIR, CLASSES, NUM_WORKERS

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [11]:
# custom modules
from modules.Dataset import get_dataloaders
from modules.Train import train
from modules.TrainFinetune import train_finetune
from modules.TrainFinetuneLayerwise import train_finetune_layerwise
from modules.Models.YOLOv1 import YOLOv1
from modules.Models.YOLOv1Dropout import YOLOv1Dropout
from modules.Models.YOLOv1Finetune import YOLOv1Finetune
from modules.Evaluation import evaluate
from modules.Inference import inference
from contextlib import contextmanager, redirect_stdout

Class and Method Declaration

In [12]:

@contextmanager
def reuse_existing_wandb_run():
    """Reuse the active sweep run inside helper functions that may call wandb.init/finish."""
    original_init = wandb.init
    original_finish = wandb.finish

    def _reuse_init(*args, **kwargs):
        return wandb.run if wandb.run is not None else original_init(*args, **kwargs)

    def _noop_finish(*args, **kwargs):
        return None

    wandb.init = _reuse_init
    wandb.finish = _noop_finish
    try:
        yield
    finally:
        wandb.init = original_init
        wandb.finish = original_finish

In [13]:

def parse_map_metrics(eval_text):
    map50_match = re.search(r"mAP@0\.50:\s*([0-9]*\.?[0-9]+)", eval_text)
    map5095_match = re.search(r"mAP@0\.50:0\.95:\s*([0-9]*\.?[0-9]+)", eval_text)

    map50 = float(map50_match.group(1)) if map50_match else None
    map5095 = float(map5095_match.group(1)) if map5095_match else None
    return {
        "val/mAP": map50,
        "val/mAP_50_95": map5095,
    }

In [14]:

def set_dropout_p(model, dropout_p):
    """Update all Dropout layers in-place. Safe even if the model has no dropout layers."""
    dropout_layers = 0
    for module in model.modules():
        if isinstance(module, torch.nn.Dropout):
            module.p = float(dropout_p)
            dropout_layers += 1
    print(f"Updated {dropout_layers} dropout layer(s) to p={float(dropout_p):.3f}")
    return dropout_layers

In [15]:
BEST_SWEEP_CKPT = None
BEST_SWEEP_RUN = None
BEST_SWEEP_MAP50 = float("-inf")
BEST_SWEEP_SUMMARY = None

In [16]:
def build_sweep_model(dropout_p):
    model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)
    set_dropout_p(model, dropout_p)

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = torch.nn.DataParallel(model)

    raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model
    return model, raw_model

In [17]:
def train_sweep():
    global BEST_SWEEP_CKPT, BEST_SWEEP_RUN, BEST_SWEEP_MAP50, BEST_SWEEP_SUMMARY

    run = wandb.init(project=SWEEP_PROJECT)
    config = wandb.config

    run_name = f"sweep_{run.id}"
    wandb.run.name = run_name
    print(f"Starting sweep run: {run_name}")

    # fresh loaders for each run
    train_loader, val_loader, _ = get_dataloaders(
        int(config.BATCH_SIZE),
        S, B, C,
        augment=bool(config.AUGMENT)
    )

    model, raw_model = build_sweep_model(config.DROPOUT_P)

    ckpt_path = os.path.join(CKPT_DIR, f"{run_name}.pth")

    with reuse_existing_wandb_run():
        raw_model = train_finetune_layerwise(
            model=model,
            raw_model=raw_model,
            train_loader=train_loader,
            val_loader=val_loader,
            S=S, B=B, C=C,
            BATCH_SIZE=int(config.BATCH_SIZE),
            EPOCHS=int(config.EPOCHS),
            LR_HEAD=float(config.LR_HEAD),
            LR_BACKBONE=float(config.LR_BACKBONE),
            WEIGHT_DECAY=float(config.WEIGHT_DECAY),
            LAMBDA_BOX=float(config.LAMBDA_BOX),
            LAMBDA_NOOBJ=float(config.LAMBDA_NOOBJ),
            RUN_NAME=run_name
        )

    torch.save(raw_model.state_dict(), ckpt_path)
    print(f"Sweep weights saved: {ckpt_path}")

    raw_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    metrics, eval_text = evaluate_with_capture(
        model=model,
        loader=val_loader,
        conf_thresh=float(config.CONF_THRESH),
        iou_thresh=NMS_IOU_THRESH
    )

    summary = {
        "val/mAP": metrics["val/mAP"],
        "val/mAP_50_95": metrics["val/mAP_50_95"],
        "ckpt_path": ckpt_path,
        "dropout_p": float(config.DROPOUT_P),
        "conf_thresh": float(config.CONF_THRESH),
        "augment": bool(config.AUGMENT),
    }

    wandb.log(summary)
    wandb.run.summary["ckpt_path"] = ckpt_path
    wandb.run.summary["eval_stdout"] = eval_text

    if metrics["val/mAP"] is not None and metrics["val/mAP"] > BEST_SWEEP_MAP50:
        BEST_SWEEP_MAP50 = metrics["val/mAP"]
        BEST_SWEEP_CKPT = ckpt_path
        BEST_SWEEP_RUN = run_name
        BEST_SWEEP_SUMMARY = {
            "run_name": run_name,
            "ckpt_path": ckpt_path,
            "val/mAP": metrics["val/mAP"],
            "val/mAP_50_95": metrics["val/mAP_50_95"],
            "config": dict(wandb.config),
        }

        with open(os.path.join(CKPT_DIR, "best_sweep_summary.json"), "w") as f:
            json.dump(BEST_SWEEP_SUMMARY, f, indent=2)

        print("New best sweep run found:")
        print(json.dumps(BEST_SWEEP_SUMMARY, indent=2))

    wandb.finish()

In [18]:

def evaluate_with_capture(model, loader, conf_thresh, iou_thresh):
    """Run evaluate() on the validation loader and parse printed mAP values."""
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        _ = evaluate(
            model=model,
            test_loader=loader,
            S=S, B=B, C=C,
            conf_thresh=float(conf_thresh),
            iou_thresh=iou_thresh
        )
    eval_text = buffer.getvalue()
    print(eval_text)
    metrics = parse_map_metrics(eval_text)
    return metrics, eval_text

In [19]:
# YOLO parameters
S = 7
B = 2
C = 20

# training parameters
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4

# loss weights
LAMBDA_BOX = 5.0
LAMBDA_NOOBJ = 0.5

# inference thresholds
CONF_THRESH = 0.30
NMS_IOU_THRESH = 0.45

In [20]:
# create all data loaders
train_loader, val_loader, test_loader = get_dataloaders(BATCH_SIZE, S, B, C)

Train set: 5717 images
Val set:   4076 images
Test set:  1747 images


#### Experiment 1 - Baseline
* Model: YOLOv1 (frozen VGG16 backbone)
* LR: 1e-3
* EPOCHS: 5
* Goal: verify model can learn and observe initial loss trend

In [21]:
# @title
RUN_NAME = "exp1_YOLOv1_lr1e-3"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS       = 5
LR           = 1e-3

In [22]:
# @title
# create model
model = YOLOv1(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)
else :
    print("not using GPUs")

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

not using GPUs


Training: Experiment 1

In [23]:
# @title
# training
raw_model = train(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)

# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

Using device: cuda


Epoch 1/5 Train:  18%|█▊        | 65/358 [00:04<00:19, 15.20it/s]


KeyboardInterrupt: 

Evaluation: Experiment 1

In [ ]:
# @title
# evaluation

# load trained weights before evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 2 - Lower Learning Rate
* Model: YOLOv1 (frozen VGG16 backbone)
* LR: 1e-3 → 1e-4
* EPOCHS: 20
* Goal: reduce overfitting seen in Experiment 1

In [ ]:
# @title
RUN_NAME   = "exp2_YOLOv1_lr1e-4"
CKPT_PATH  = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS       = 20
LR           = 1e-4

In [ ]:
# @title
# create model
model = YOLOv1(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

Training : Experiment 2

In [ ]:
# @title
# training
raw_model = train(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)

# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

Evaluation: Experiment 2

In [ ]:
# @title
# evaluation

# load trained weights before evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 3 - Dropout Regularisation
* Model: YOLOv1Dropout (frozen VGG16 backbone)
* LR: 1e-4
* EPOCHS: 20
* Dropout: p=0.5 added in head
* Goal: further reduce overfitting with dropout regularisation

In [ ]:
# @title
RUN_NAME  = "exp3_Dropout_lr1e-4"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR = 1e-4

In [ ]:
# @title
# create model
model = YOLOv1Dropout(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

Training: Experiment 3

In [ ]:
# @title

# training
raw_model = train(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS, LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")


Evaluation: Experiment 3

In [ ]:
# @title
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 4 - VGG16 Fine-tuning with Early Stopping
* Model: YOLOv1Finetune (last 2 conv layers of VGG16 unfrozen)
* LR: 1e-4
* EPOCHS: 20 (Early Stopping: patience=5)
* Dropout: p=0.5 in head
* Goal: improve feature extraction by fine-tuning backbone on VOC dataset

In [ ]:
# @title
RUN_NAME  = "exp4_Finetune_lr1e-4"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR = 1e-4

In [ ]:
# @title
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

Training: Experiment 4

In [ ]:
# @title
# training
raw_model = train_finetune(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

Evaluation: Experiment 4

In [ ]:
# @title
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 5 - Layer-wise Learning Rate
* Model: YOLOv1Finetune (last 2 conv layers of VGG16 unfrozen)
* LR head: 1e-4
* LR backbone: 1e-5
* EPOCHS: 20 (Early Stopping: patience=5)
* Dropout: p=0.5 in head
* Goal: more stable fine-tuning with smaller backbone LR

In [ ]:
# @title
RUN_NAME  = "exp5_Finetune_lrH1e-4_lrB1e-5"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR_HEAD=1e-4
LR_BACKBONE=1e-5

In [ ]:
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model


Training: Experiment 5

In [ ]:
# training
raw_model = train_finetune_layerwise(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR_HEAD=LR_HEAD,
    LR_BACKBONE=LR_BACKBONE,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

Evaluation: Experiment 5

In [ ]:
# @title
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 6 - Layer-wise LR Tuning
* Model: YOLOv1Finetune (last 2 conv layers of VGG16 unfrozen)
* LR head: 1e-4
* LR backbone: 5e-5 (increased from exp5)
* EPOCHS: 20 (Early Stopping: patience=5)
* Dropout: p=0.5 in head
* Goal: find better backbone LR between exp4 (1e-4) and exp5 (1e-5)

In [ ]:
RUN_NAME = "exp6_Finetune_lrH1e-4_lrB5e-5"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR_HEAD=1e-4
LR_BACKBONE= 5e-5

In [ ]:
# @title
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model


Training: Experiment 6

In [ ]:
# @title
# training
raw_model = train_finetune_layerwise(
    model = model,
    raw_model = raw_model,
    train_loader = train_loader,
    val_loader = val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE = BATCH_SIZE,
    EPOCHS = EPOCHS,
    LR_HEAD = LR_HEAD,
    LR_BACKBONE = LR_BACKBONE,
    WEIGHT_DECAY = WEIGHT_DECAY,
    LAMBDA_BOX = LAMBDA_BOX,
    LAMBDA_NOOBJ = LAMBDA_NOOBJ,
    RUN_NAME = RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")


Evluation: Experiment 6

In [ ]:
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 7 - Data Augmentation
* Model: YOLOv1Finetune (last 2 conv layers of VGG16 unfrozen)
* LR: 1e-4
* EPOCHS: 20 (Early Stopping: patience=5)
* Dropout: p=0.5 in head
* Augmentation: ColorJitter (brightness, contrast, saturation, hue)
* Goal: reduce overfitting with colour augmentation on training set

In [ ]:
RUN_NAME = "exp7_Finetune_Aug_lr1e-4"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR = 1e-4

In [ ]:
# create data loaders with augmentation
train_loader, val_loader, test_loader = get_dataloaders(BATCH_SIZE, S, B, C, augment=True)

In [ ]:
# @title
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model


Training: Experiment 7

In [ ]:
# @title
# training
raw_model = train_finetune(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")


Evaluation: Experiment 7

In [ ]:
# @title
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 8 - Automated Hyperparameter Tuning with WandB Sweeps
This section adds a **Bayesian WandB Sweep** on top of the strongest manual setup:
* Model: `YOLOv1Finetune`
* Optimiser style: **layer-wise learning rates**
* Sweep metric: **validation mAP@0.50**
* Search space: `LR_HEAD`, `LR_BACKBONE`, `WEIGHT_DECAY`, `LAMBDA_BOX`, `LAMBDA_NOOBJ`, `DROPOUT_P`, `CONF_THRESH`

Notes:
* This reuses the existing `train_finetune_layerwise()` training function.
* A small WandB patch is included so the sweep run is reused even if your training helpers already call `wandb.init()` or `wandb.finish()`.
* `DROPOUT_P` is applied by updating any `torch.nn.Dropout` layers found in the model.

Weights and Biases Sweep: Results

In [ ]:
# @title

SWEEP_PROJECT = "yolo-object-detection"
# How many runs (trials) the agent will execute. Increase for better search, decrease for time/compute limits.
SWEEP_COUNT = 20  # reduce this if Colab time is tight

sweep_config = {
    # Bayesian optimisation over the parameters below
    "method": "bayes",

    # The scalar to maximise across sweep trials
    "metric": {"name": "val/mAP", "goal": "maximize"},

    "parameters": {
        # Fixed knobs (kept constant across all trials)
        "EPOCHS": {"value": 20},
        "BATCH_SIZE": {"value": BATCH_SIZE},
        "AUGMENT": {"value": False},

        # Learning rates: search on a log scale (LRs typically vary by orders of magnitude)
        "LR_HEAD": {
            "min": 1e-5,
            "max": 1e-3,
            "distribution": "log_uniform_values",
        },
        "LR_BACKBONE": {
            "min": 1e-6,
            "max": 1e-4,
            "distribution": "log_uniform_values",
        },

        # Weight decay: also varies best on a log scale
        "WEIGHT_DECAY": {
            "min": 1e-5,
            "max": 1e-3,
            "distribution": "log_uniform_values",
        },

        # YOLO loss weights: linear ranges are OK (these are already in human-scale ranges)
        "LAMBDA_BOX": {
            "min": 2.0,
            "max": 10.0,
        },
        "LAMBDA_NOOBJ": {
            "min": 0.1,
            "max": 1.0,
        },

        # Dropout probability applied to any `torch.nn.Dropout` layers found in the model
        # (If the model has no Dropout layers, this won't change anything.)
        "DROPOUT_P": {
            "min": 0.3,
            "max": 0.7,
        },

        # Confidence threshold used during evaluation to filter predictions.
        # NOTE: This is an *evaluation-time* knob, not training-time. Optimising it can inflate mAP by tuning the
        # decision threshold; keep it fixed if you want strict apples-to-apples model comparisons.
        "CONF_THRESH": {
            "min": 0.2,
            "max": 0.5,
        },
    },
}

print(json.dumps(sweep_config, indent=2))

In [ ]:
# @title
sweep_id = wandb.sweep(sweep_config, project=SWEEP_PROJECT)
print(f"Sweep ID: {sweep_id}")
wandb.agent(sweep_id, function=train_sweep, count=SWEEP_COUNT)

print("\nBest sweep summary:")
print(json.dumps(BEST_SWEEP_SUMMARY, indent=2) if BEST_SWEEP_SUMMARY else "No successful sweep result captured.")

In [ ]:
# @title
import pandas as pd
import wandb

api = wandb.Api()

# replace with your actual values
sweep = api.sweep("y-benjamin_pc-city-st-george-s-university-of-london/yolo-object-detection/8ljoiuem")

rows = []
for run in sweep.runs:
    cfg = {k: v for k, v in run.config.items() if not k.startswith("_")}
    summ = dict(run.summary)

    rows.append({
        "run_name": run.name,
        "state": run.state,
        "val/mAP": summ.get("val/mAP"),
        "val/mAP_50_95": summ.get("val/mAP_50_95"),
        "train/loss": summ.get("train/loss"),
        "val/loss": summ.get("val/loss"),
        "precision": summ.get("val/precision"),
        "recall": summ.get("val/recall"),
        "LR_HEAD": cfg.get("LR_HEAD"),
        "LR_BACKBONE": cfg.get("LR_BACKBONE"),
        "LAMBDA_BOX": cfg.get("LAMBDA_BOX"),
        "LAMBDA_NOOBJ": cfg.get("LAMBDA_NOOBJ"),
        "WEIGHT_DECAY": cfg.get("WEIGHT_DECAY"),
        "DROPOUT_P": cfg.get("DROPOUT_P"),
        "CONF_THRESH": cfg.get("CONF_THRESH"),
        "BATCH_SIZE": cfg.get("BATCH_SIZE"),
        "AUGMENT": cfg.get("AUGMENT"),
    })

df = pd.DataFrame(rows)

# best by val/mAP
df_map = df.sort_values("val/mAP", ascending=False)

# best by stricter localisation metric
df_map5095 = df.sort_values("val/mAP_50_95", ascending=False)

print("Top 10 by val/mAP")
print(df_map.head(10).to_string(index=False))

print("\nTop 10 by val/mAP_50_95")
print(df_map5095.head(10).to_string(index=False))

df.to_csv("sweep_results_full.csv", index=False)

In [ ]:
# @title
import wandb
import pandas as pd

ENTITY = "y-benjamin_pc-city-st-george-s-university-of-london"
PROJECT = "yolo-object-detection"
TOP_K = 10

api = wandb.Api()
runs = api.runs(f"{ENTITY}/{PROJECT}")

rows = []

for run in runs:
    summary = run.summary or {}
    config = run.config or {}

    rows.append({
        "run_name": run.name,
        "state": run.state,
        "val/mAP": summary.get("val/mAP"),
        "val/mAP_50_95": summary.get("val/mAP_50_95"),
        "train/loss": summary.get("train/loss"),
        "val/loss": summary.get("val/loss"),
        "precision": summary.get("precision"),
        "recall": summary.get("recall"),
        "LR_HEAD": config.get("LR_HEAD"),
        "LR_BACKBONE": config.get("LR_BACKBONE"),
        "LAMBDA_BOX": config.get("LAMBDA_BOX"),
        "LAMBDA_NOOBJ": config.get("LAMBDA_NOOBJ"),
        "WEIGHT_DECAY": config.get("WEIGHT_DECAY"),
        "DROPOUT_P": config.get("DROPOUT_P"),
        "CONF_THRESH": config.get("CONF_THRESH"),
        "BATCH_SIZE": config.get("BATCH_SIZE"),
        "AUGMENT": config.get("AUGMENT"),
    })

df = pd.DataFrame(rows)

# top runs by val/mAP
top_map = (
    df.dropna(subset=["val/mAP"])
      .sort_values("val/mAP", ascending=False)
      .head(TOP_K)
)

print(f"Top {TOP_K} by val/mAP")
print(top_map.to_string(index=False))

# top runs by val/mAP_50_95
top_map5095 = (
    df.dropna(subset=["val/mAP_50_95"])
      .sort_values("val/mAP_50_95", ascending=False)
      .head(TOP_K)
)

print("\n" + "="*100 + "\n")
print(f"Top {TOP_K} by val/mAP_50_95")
print(top_map5095.to_string(index=False))

In [ ]:
# @title


ENTITY = "y-benjamin_pc-city-st-george-s-university-of-london"
PROJECT = "yolo-object-detection"

# folder where outputs will be saved
OUTPUT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Learning for Image Analysis/YOLO-object-detection/graphs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# metrics to export if present
LOSS_METRICS = [
    "train_loss",
    "val_loss",
    "box_loss",
    "obj_loss",
    "noobj_loss",
    "cls_loss",
]

# if you only want runs whose names contain something, set it here
# example: RUN_NAME_FILTER = "exp"
RUN_NAME_FILTER = None

# if True, save raw CSV history for each run
SAVE_CSV = True


def safe_filename(text: str) -> str:
    """Make a filename safe for Windows/macOS/Linux."""
    bad = '<>:"/\\|?*'
    for ch in bad:
        text = text.replace(ch, "_")
    return text.strip().replace(" ", "_")


def get_epoch_or_step(df: pd.DataFrame) -> pd.Series:
    """Use epoch if present, else _step, else dataframe index."""
    if "epoch" in df.columns:
        return df["epoch"]
    if "_step" in df.columns:
        return df["_step"]
    return pd.Series(df.index, index=df.index)


def export_run_history(run, output_dir: Path) -> pd.DataFrame | None:
    """Download one run's history and optionally save CSV."""
    try:
        df = run.history(samples=100000)
    except Exception as e:
        print(f"[SKIP] Could not load history for {run.name}: {e}")
        return None

    if df is None or df.empty:
        print(f"[SKIP] No history for {run.name}")
        return None

    run_slug = safe_filename(f"{run.name}_{run.id}")
    run_dir = output_dir / run_slug
    run_dir.mkdir(parents=True, exist_ok=True)

    if SAVE_CSV:
        csv_path = run_dir / "history.csv"
        df.to_csv(csv_path, index=False)

    return df


def plot_individual_losses(run, df: pd.DataFrame, output_dir: Path) -> None:
    """Save one PNG per loss metric if present."""
    run_slug = safe_filename(f"{run.name}_{run.id}")
    run_dir = output_dir / run_slug
    x = get_epoch_or_step(df)

    for metric in LOSS_METRICS:
        if metric not in df.columns:
            continue

        metric_df = pd.DataFrame({"x": x, "y": df[metric]}).dropna()
        if metric_df.empty:
            continue

        plt.figure(figsize=(8, 5))
        plt.plot(metric_df["x"], metric_df["y"])
        plt.xlabel("Epoch" if "epoch" in df.columns else "Step")
        plt.ylabel(metric)
        plt.title(f"{run.name} - {metric}")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(run_dir / f"{metric}.png", dpi=200)
        plt.close()


def plot_train_vs_val(run, df: pd.DataFrame, output_dir: Path) -> None:
    """Save a combined train vs val loss graph if both exist."""
    if "train_loss" not in df.columns or "val_loss" not in df.columns:
        return

    run_slug = safe_filename(f"{run.name}_{run.id}")
    run_dir = output_dir / run_slug
    x = get_epoch_or_step(df)

    pair_df = pd.DataFrame({
        "x": x,
        "train_loss": df["train_loss"],
        "val_loss": df["val_loss"],
    }).dropna(how="all")

    if pair_df.empty:
        return

    plt.figure(figsize=(8, 5))
    if pair_df["train_loss"].notna().any():
        plt.plot(pair_df["x"], pair_df["train_loss"], label="train_loss")
    if pair_df["val_loss"].notna().any():
        plt.plot(pair_df["x"], pair_df["val_loss"], label="val_loss")

    plt.xlabel("Epoch" if "epoch" in df.columns else "Step")
    plt.ylabel("Loss")
    plt.title(f"{run.name} - Train vs Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(run_dir / "train_vs_val_loss.png", dpi=200)
    plt.close()


def build_summary_row(run, df: pd.DataFrame) -> dict:
    """Create a summary row for quick comparison across runs."""
    row = {
        "run_name": run.name,
        "run_id": run.id,
        "state": getattr(run, "state", None),
        "url": getattr(run, "url", None),
    }

    summary = getattr(run, "summary", {}) or {}

    # W&B summary keys vary depending on logging style
    candidate_keys = [
        "val_loss",
        "train_loss",
        "box_loss",
        "obj_loss",
        "noobj_loss",
        "cls_loss",
        "val/mAP",
        "val/mAP50",
        "metrics/mAP50",
        "mAP@0.50",
        "mAP50",
    ]

    for key in candidate_keys:
        row[key] = summary.get(key, None)

    # fallback: final history values if summary is missing
    for metric in LOSS_METRICS:
        if row.get(metric) is None and metric in df.columns:
            non_null = df[metric].dropna()
            row[metric] = non_null.iloc[-1] if not non_null.empty else None

    return row


def plot_compare_metric_across_runs(summary_df: pd.DataFrame, metric: str, output_dir: Path) -> None:
    """Plot one bar chart across runs for a given metric."""
    if metric not in summary_df.columns:
        return

    temp = summary_df[["run_name", metric]].dropna()
    if temp.empty:
        return

    # sort so the chart is easier to read
    temp = temp.sort_values(by=metric, ascending=True)

    plt.figure(figsize=(10, max(5, len(temp) * 0.35)))
    plt.barh(temp["run_name"], temp[metric])
    plt.xlabel(metric)
    plt.ylabel("Run")
    plt.title(f"{metric} across runs")
    plt.tight_layout()
    plt.savefig(output_dir / f"compare_{safe_filename(metric)}.png", dpi=200)
    plt.close()


def main():
    api = wandb.Api()
    runs = api.runs(f"{ENTITY}/{PROJECT}")

    summary_rows = []
    processed = 0

    for run in runs:
        if RUN_NAME_FILTER and RUN_NAME_FILTER.lower() not in (run.name or "").lower():
            continue

        print(f"[INFO] Processing run: {run.name} ({run.id})")
        df = export_run_history(run, OUTPUT_DIR)
        if df is None:
            continue

        plot_individual_losses(run, df, OUTPUT_DIR)
        plot_train_vs_val(run, df, OUTPUT_DIR)

        summary_rows.append(build_summary_row(run, df))
        processed += 1

    if not summary_rows:
        print("[DONE] No runs matched.")
        return

    summary_df = pd.DataFrame(summary_rows)
    summary_path = OUTPUT_DIR / "run_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    # comparison charts across runs
    for metric in ["val_loss", "train_loss", "val/mAP", "val/mAP50", "mAP@0.50", "mAP50"]:
        plot_compare_metric_across_runs(summary_df, metric, OUTPUT_DIR)

    print(f"[DONE] Processed {processed} runs.")
    print(f"[DONE] Output saved to: {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    main()

Experiment 9

In [24]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import wandb

print("CUDA:", torch.cuda.is_available())
print("MODEL DEVICE:", next(model.parameters()).device)

x = torch.randn(1, 3, 224, 224).to(DEVICE)
print("TEST TENSOR DEVICE:", x.device)

FINAL_RETRAIN_SEEDS = [SEED, SEED + 1, SEED + 2]

# IMPORTANT: no double .pth
TARGET_SWEEP_RUN = "sweep_8nxmzivh"
SELECTED_SWEEP_CKPT = os.path.join(CKPT_DIR, f"{TARGET_SWEEP_RUN}.pth")

FINAL_SWEEP_CONFIG = {
    "LR_HEAD": 2.73e-2,
    "LR_BACKBONE": 8.1e-5,
    "WEIGHT_DECAY": 1e-4,
    "LAMBDA_BOX": 0.05,
    "LAMBDA_NOOBJ": 0.5,
    "DROPOUT_P": 0.3,
    "CONF_THRESH": 0.5,
    "BATCH_SIZE": 16,
    "AUGMENT": True,
    "EPOCHS": 20,
}

FINAL_RESULTS_CSV = os.path.join(CKPT_DIR, "exp9_final_retrain_results.csv")
FINAL_SUMMARY_JSON = os.path.join(CKPT_DIR, "exp9_final_retrain_summary.json")

BEST_FINAL_CKPT = None
BEST_FINAL_RUN_NAME = None
BEST_FINAL_SUMMARY = None
FINAL_RETRAIN_RESULTS = []

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Using forced sweep:")
print(FINAL_SWEEP_CONFIG)
print("Checkpoint:", SELECTED_SWEEP_CKPT)
print("Checkpoint exists:", os.path.exists(SELECTED_SWEEP_CKPT))

CUDA: True
MODEL DEVICE: cuda:0
TEST TENSOR DEVICE: cuda:0
Using forced sweep:
{'LR_HEAD': 0.0273, 'LR_BACKBONE': 8.1e-05, 'WEIGHT_DECAY': 0.0001, 'LAMBDA_BOX': 0.05, 'LAMBDA_NOOBJ': 0.5, 'DROPOUT_P': 0.3, 'CONF_THRESH': 0.5, 'BATCH_SIZE': 16, 'AUGMENT': True, 'EPOCHS': 20}
Checkpoint: /content/drive/MyDrive/Colab Notebooks/Education/Deep Learning for Image Analysis/YOLO-object-detection/checkpoints/sweep_8nxmzivh.pth
Checkpoint exists: True


In [25]:
import os
import torch
import wandb

os.makedirs(CKPT_DIR, exist_ok=True)
FINAL_RETRAIN_RESULTS = []

for seed in FINAL_RETRAIN_SEEDS:
    run_name = f"exp9_seed{seed}"
    ckpt_path = os.path.join(CKPT_DIR, f"{run_name}.pth")

    print("=" * 80)
    print(f"Starting run: {run_name}")
    print(f"Warm-starting from: {SELECTED_SWEEP_CKPT}")

    set_all_seeds(seed)

    train_loader, val_loader, test_loader = get_dataloaders(
        FINAL_SWEEP_CONFIG["BATCH_SIZE"],
        S, B, C,
        augment=FINAL_SWEEP_CONFIG["AUGMENT"]
    )

    model, raw_model = build_sweep_model(FINAL_SWEEP_CONFIG["DROPOUT_P"])

    # LOAD THE BEST SWEEP CHECKPOINT
    if not os.path.exists(SELECTED_SWEEP_CKPT):
        raise FileNotFoundError(f"Sweep checkpoint not found: {SELECTED_SWEEP_CKPT}")

    state_dict = torch.load(SELECTED_SWEEP_CKPT, map_location=DEVICE)
    raw_model.load_state_dict(state_dict)
    print(f"Loaded sweep checkpoint: {SELECTED_SWEEP_CKPT}")

    run = wandb.init(
        entity="y-benjamin_pc-city-st-george-s-university-of-london",
        project="yolo-object-detection",
        name=run_name,
        config={
            "seed": seed,
            "architecture": "YOLOv1Finetune_VGG16",
            "dataset": "Pascal VOC 2012",
            "source_sweep_ckpt": SELECTED_SWEEP_CKPT,
            "S": S,
            "B": B,
            "C": C,
            "batch_size": FINAL_SWEEP_CONFIG["BATCH_SIZE"],
            "epochs": FINAL_SWEEP_CONFIG["EPOCHS"],
            "augment": FINAL_SWEEP_CONFIG["AUGMENT"],
            "dropout_p": FINAL_SWEEP_CONFIG["DROPOUT_P"],
            "lr_head": FINAL_SWEEP_CONFIG["LR_HEAD"],
            "lr_backbone": FINAL_SWEEP_CONFIG["LR_BACKBONE"],
            "weight_decay": FINAL_SWEEP_CONFIG["WEIGHT_DECAY"],
            "lambda_box": FINAL_SWEEP_CONFIG["LAMBDA_BOX"],
            "lambda_noobj": FINAL_SWEEP_CONFIG["LAMBDA_NOOBJ"],
            "conf_thresh": FINAL_SWEEP_CONFIG["CONF_THRESH"],
            "nms_iou_thresh": NMS_IOU_THRESH,
        }
    )

    try:
        raw_model = train_finetune_layerwise(
            model=model,
            raw_model=raw_model,
            train_loader=train_loader,
            val_loader=val_loader,
            S=S, B=B, C=C,
            BATCH_SIZE=FINAL_SWEEP_CONFIG["BATCH_SIZE"],
            EPOCHS=FINAL_SWEEP_CONFIG["EPOCHS"],
            LR_HEAD=FINAL_SWEEP_CONFIG["LR_HEAD"],
            LR_BACKBONE=FINAL_SWEEP_CONFIG["LR_BACKBONE"],
            WEIGHT_DECAY=FINAL_SWEEP_CONFIG["WEIGHT_DECAY"],
            LAMBDA_BOX=FINAL_SWEEP_CONFIG["LAMBDA_BOX"],
            LAMBDA_NOOBJ=FINAL_SWEEP_CONFIG["LAMBDA_NOOBJ"],
            RUN_NAME=run_name,
            CONF_THRESH=FINAL_SWEEP_CONFIG["CONF_THRESH"],
            NMS_IOU_THRESH=NMS_IOU_THRESH,
        )

        torch.save(raw_model.state_dict(), ckpt_path)
        print(f"Successfully saved: {ckpt_path}")

        model.eval()

        val_metrics, _ = evaluate_with_capture(
            model=model,
            loader=val_loader,
            conf_thresh=FINAL_SWEEP_CONFIG["CONF_THRESH"],
            iou_thresh=NMS_IOU_THRESH
        )

        test_metrics, _ = evaluate_with_capture(
            model=model,
            loader=test_loader,
            conf_thresh=FINAL_SWEEP_CONFIG["CONF_THRESH"],
            iou_thresh=NMS_IOU_THRESH
        )

        wandb.log({
            "val/mAP": val_metrics["val/mAP"],
            "val/mAP_50_95": val_metrics["val/mAP_50_95"],
            "test/mAP": test_metrics["val/mAP"],
            "test/mAP_50_95": test_metrics["val/mAP_50_95"],
        })

        wandb.summary["source_sweep_ckpt"] = SELECTED_SWEEP_CKPT
        wandb.summary["ckpt_path"] = ckpt_path
        wandb.summary["final_val_mAP"] = val_metrics["val/mAP"]
        wandb.summary["final_val_mAP_50_95"] = val_metrics["val/mAP_50_95"]
        wandb.summary["final_test_mAP"] = test_metrics["val/mAP"]
        wandb.summary["final_test_mAP_50_95"] = test_metrics["val/mAP_50_95"]

        FINAL_RETRAIN_RESULTS.append({
            "run_name": run_name,
            "seed": seed,
            "source_sweep_ckpt": SELECTED_SWEEP_CKPT,
            "ckpt_path": ckpt_path,
            "val/mAP": val_metrics["val/mAP"],
            "val/mAP_50_95": val_metrics["val/mAP_50_95"],
            "test/mAP": test_metrics["val/mAP"],
            "test/mAP_50_95": test_metrics["val/mAP_50_95"],
        })

        print(
            f"{run_name} | "
            f"val/mAP: {val_metrics['val/mAP']} | "
            f"val/mAP_50_95: {val_metrics['val/mAP_50_95']} | "
            f"test/mAP: {test_metrics['val/mAP']} | "
            f"test/mAP_50_95: {test_metrics['val/mAP_50_95']}"
        )

    finally:
        wandb.finish()

Starting run: exp9_seed42
Warm-starting from: /content/drive/MyDrive/Colab Notebooks/Education/Deep Learning for Image Analysis/YOLO-object-detection/checkpoints/sweep_8nxmzivh.pth
Train set: 5717 images
Val set:   4076 images
Test set:  1747 images
Updated 1 dropout layer(s) to p=0.300
Loaded sweep checkpoint: /content/drive/MyDrive/Colab Notebooks/Education/Deep Learning for Image Analysis/YOLO-object-detection/checkpoints/sweep_8nxmzivh.pth


Using device: cuda


Epoch 1/20 ValLoss: 100%|██████████| 255/255 [00:13<00:00, 19.02it/s]


Epoch 001/20 | train: 4.9675 | val_loss: 3.0044 | val_mAP: 0.0215 | val_mAP_50_95: 0.0052
Best model at epoch 1 | val/mAP: 0.0215 | val_loss: 3.0044


Epoch 2/20 ValLoss: 100%|██████████| 255/255 [00:13<00:00, 18.84it/s]


...

Epoch 3/20 ValLoss: 100%|██████████| 255/255 [00:13<00:00, 18.62it/s]


Epoch 003/20 | train: 2.2736 | val_loss: 2.9997 | val_mAP: 0.029 | val_mAP_50_95: 0.007
Best model at epoch 3 | val/mAP: 0.029 | val_loss: 2.9997


Epoch 4/20 ValLoss: 100%|██████████| 255/255 [00:13<00:00, 18.67it/s]


box_loss,█▂▁
cls_loss,█▂▁
epoch,▁▅█
noobj_loss,█▁▁
obj_loss,█▁▁
train_loss,█▁▁
val/mAP,▁▇█
val/mAP_50_95,▁██
val_loss,▁▃█
box_loss,0.32525
cls_loss,1.84521


KeyboardInterrupt: 

In [ ]:
FINAL_RETRAIN_DF = pd.DataFrame(FINAL_RETRAIN_RESULTS)
display(FINAL_RETRAIN_DF)

if FINAL_RETRAIN_DF.empty:
    raise RuntimeError("No final retraining results were collected.")

FINAL_RETRAIN_DF.to_csv(FINAL_RESULTS_CSV, index=False)

for col in ["val/mAP", "test/mAP", "val/mAP_50_95", "test/mAP_50_95"]:
    FINAL_RETRAIN_DF[col] = pd.to_numeric(FINAL_RETRAIN_DF[col], errors="coerce")

best_final_idx = FINAL_RETRAIN_DF["val/mAP"].idxmax()
best_final_row = FINAL_RETRAIN_DF.loc[best_final_idx]

BEST_FINAL_CKPT = best_final_row["ckpt_path"]
BEST_FINAL_RUN_NAME = best_final_row["run_name"]

BEST_FINAL_SUMMARY = {
    "best_final_run_name": BEST_FINAL_RUN_NAME,
    "best_final_ckpt": BEST_FINAL_CKPT,
    "best_final_seed": int(best_final_row["seed"]),
    "best_final_val_mAP": float(best_final_row["val/mAP"]),
    "best_final_test_mAP": float(best_final_row["test/mAP"]),
    "mean_val_mAP": float(FINAL_RETRAIN_DF["val/mAP"].mean()),
    "std_val_mAP": float(FINAL_RETRAIN_DF["val/mAP"].std(ddof=0)),
    "mean_test_mAP": float(FINAL_RETRAIN_DF["test/mAP"].mean()),
    "std_test_mAP": float(FINAL_RETRAIN_DF["test/mAP"].std(ddof=0)),
    "mean_val_mAP_50_95": float(FINAL_RETRAIN_DF["val/mAP_50_95"].mean()),
    "std_val_mAP_50_95": float(FINAL_RETRAIN_DF["val/mAP_50_95"].std(ddof=0)),
    "mean_test_mAP_50_95": float(FINAL_RETRAIN_DF["test/mAP_50_95"].mean()),
    "std_test_mAP_50_95": float(FINAL_RETRAIN_DF["test/mAP_50_95"].std(ddof=0)),
    "final_retrain_seeds": FINAL_RETRAIN_SEEDS,
    "source_sweep_run_name": TARGET_SWEEP_RUN,
    "source_sweep_ckpt": SELECTED_SWEEP_CKPT,
    "source_sweep_config": FINAL_SWEEP_CONFIG,
}

with open(FINAL_SUMMARY_JSON, "w") as f:
    json.dump(BEST_FINAL_SUMMARY, f, indent=2)

print("Final retraining summary:")
print(json.dumps(BEST_FINAL_SUMMARY, indent=2))

In [ ]:
FINAL_RETRAIN_DF = pd.DataFrame(FINAL_RETRAIN_RESULTS)
display(FINAL_RETRAIN_DF)

if FINAL_RETRAIN_DF.empty:
    raise RuntimeError("No final retraining results were collected.")

FINAL_RETRAIN_DF.to_csv(FINAL_RESULTS_CSV, index=False)

# --- Safe numeric conversion ---
for col in ["val/mAP", "test/mAP", "val/mAP_50_95", "test/mAP_50_95"]:
    if col in FINAL_RETRAIN_DF.columns:
        FINAL_RETRAIN_DF[col] = pd.to_numeric(FINAL_RETRAIN_DF[col], errors="coerce")

# --- Best run ---
best_final_idx = FINAL_RETRAIN_DF["val/mAP"].idxmax()
best_final_row = FINAL_RETRAIN_DF.loc[best_final_idx]

BEST_FINAL_CKPT = best_final_row["ckpt_path"]
BEST_FINAL_RUN_NAME = best_final_row["run_name"]

# --- Safe stats ---
def safe_stat(col, fn):
    return float(fn(FINAL_RETRAIN_DF[col])) if col in FINAL_RETRAIN_DF.columns else None

BEST_FINAL_SUMMARY = {
    "best_final_run_name": BEST_FINAL_RUN_NAME,
    "best_final_ckpt": BEST_FINAL_CKPT,
    "best_final_seed": int(best_final_row["seed"]),
    "best_final_val_mAP": float(best_final_row["val/mAP"]),
    "best_final_test_mAP": float(best_final_row["test/mAP"]),

    "mean_val_mAP": safe_stat("val/mAP", pd.Series.mean),
    "std_val_mAP": safe_stat("val/mAP", lambda x: x.std(ddof=0)),

    "mean_test_mAP": safe_stat("test/mAP", pd.Series.mean),
    "std_test_mAP": safe_stat("test/mAP", lambda x: x.std(ddof=0)),

    "mean_val_mAP_50_95": safe_stat("val/mAP_50_95", pd.Series.mean),
    "std_val_mAP_50_95": safe_stat("val/mAP_50_95", lambda x: x.std(ddof=0)),

    "mean_test_mAP_50_95": safe_stat("test/mAP_50_95", pd.Series.mean),
    "std_test_mAP_50_95": safe_stat("test/mAP_50_95", lambda x: x.std(ddof=0)),

    "final_retrain_seeds": FINAL_RETRAIN_SEEDS,

    "source_sweep_run_name": TARGET_SWEEP_RUN,
    "source_sweep_ckpt": SELECTED_SWEEP_CKPT,
    "source_sweep_config": FINAL_SWEEP_CONFIG,
}

with open(FINAL_SUMMARY_JSON, "w") as f:
    json.dump(BEST_FINAL_SUMMARY, f, indent=2)

print("Final retraining summary:")
print(json.dumps(BEST_FINAL_SUMMARY, indent=2))

Experiment 10 Qualitative Error and Analysis

In [ ]:
from config import IMG_DIR

# create data loaders
train_loader, val_loader, test_loader = get_dataloaders(BATCH_SIZE, S, B, C)

# get 10 images from test set
test_ids = [test_loader.dataset.dataset.img_ids[i] for i in range(10)]

# inference threshold (higher than eval threshold)
INFERENCE_CONF_THRESH = 0.80

# create and load best model exp4
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)
model.load_state_dict(torch.load(
    f"{CKPT_DIR}/exp4_Finetune_lr1e-4.pth",
    map_location=DEVICE,
    weights_only=True
))
model.eval()



# run inference on 5 test images
for img_id in test_ids:
    img_path = f"{IMG_DIR}/{img_id}.jpg"
    inference(
        model=model,
        img_path=img_path,
        S=S, B=B, C=C,
        conf_thresh=INFERENCE_CONF_THRESH,
        iou_thresh=NMS_IOU_THRESH
    )